# E-Nose: Dataset Curation — Feature Scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Scarage1/E-Nose/blob/main/notebooks/dataset_curation.ipynb)

This notebook preprocesses raw gas sensor CSV files collected by the E-Nose firmware:
1. **Analyse** raw data distributions (scatter matrix, histograms)
2. **Choose** a preprocessing method for each column (drop / normalise / standardise)
3. **Apply** preprocessing and inspect the result
4. **Export** scaled CSV files ready for upload to Edge Impulse

### Column order (raw CSVs)
```
timestamp, temp, humd, pres, co2, voc1, voc2, no2, eth, co
```

### Instructions
1. Upload your raw CSV files to `/content/dataset/` (or update `DATASET_PATH` below).
2. Run all cells in order.
3. Download `out.zip` when finished.
4. Copy the printed **Mins** and **Ranges** into the inference firmware.

---
_References_  
[1] LeCun et al., "Efficient BackProp" — http://yann.lecun.com/exdb/publis/pdf/lecun-98b.pdf  
[2] Feature scaling overview — https://www.atoti.io/articles/when-to-perform-a-feature-scaling/

In [ ]:
import csv
import os
import shutil

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
### ── Settings ──────────────────────────────────────────────────────────────
HOME_PATH    = "/content"          # Working directory (Colab default)
DATASET_PATH = "/content/dataset"  # Upload your raw .csv files here
OUT_PATH     = "/content/out"      # Output directory (will be recreated)
OUT_ZIP      = "/content/out.zip"  # Path for zipped output

# Preprocessing codes (do not change)
PREP_DROP = -1   # Drop this column entirely
PREP_NONE =  0   # No preprocessing
PREP_STD  =  1   # Standardisation  → mean=0, var=1  (use for Gaussian data)
PREP_NORM =  2   # Normalisation     → [0, 1]         (use for non-Gaussian data)

print("Settings loaded.")

## Step 1 — Load and Analyse the Raw Data

In [ ]:
### Read all .csv files in DATASET_PATH into one combined array

header    = None
raw_data  = []
num_lines = []    # How many data rows came from each file
filenames = []    # Original filenames (used when writing output)

for filename in sorted(os.listdir(DATASET_PATH)):
    filepath = os.path.join(DATASET_PATH, filename)
    if not os.path.isfile(filepath):
        continue

    with open(filepath, newline="") as f:
        reader = csv.reader(f)
        for line_count, line in enumerate(reader):

            if line_count == 0:              # Header row
                if header is None:
                    header = line
                if header == line:
                    num_lines.append(0)
                    filenames.append(filename)
                else:
                    print(f"Warning: header mismatch in {filename} — skipping file")
                    break
            else:                            # Data rows
                if len(line) == len(header):
                    raw_data.append(line)
                    num_lines[-1] += 1
                else:
                    print(f"Warning: column count mismatch in {filename} line {line_count} — skipping")

raw_data = np.array(raw_data, dtype=float)

print(f"Loaded {len(filenames)} file(s)")
print(f"Header : {header}")
print(f"Shape  : {raw_data.shape}  (rows × columns)")
assert len(num_lines) == len(filenames)

In [ ]:
### Scatter matrix — look for inter-feature correlation
df = pd.DataFrame(raw_data, columns=header)
_ = pd.plotting.scatter_matrix(df, figsize=(16, 16), alpha=0.3, diagonal="hist")
plt.suptitle("Scatter Matrix — Raw Data", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
### Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.matshow(df.corr(), cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(im)
ax.set_xticks(range(len(header)))
ax.set_xticklabels(header, rotation=45, ha="left")
ax.set_yticks(range(len(header)))
ax.set_yticklabels(header)
ax.set_title("Correlation Matrix — Raw Data", pad=20)
plt.tight_layout()
plt.show()

In [ ]:
### Histograms — check distribution shape (Gaussian vs non-Gaussian)
n = len(header)
fig, axs = plt.subplots(1, n, figsize=(3 * n, 3))
for i, name in enumerate(header):
    axs[i].hist(raw_data[:, i], bins=30, color="steelblue", edgecolor="white")
    axs[i].set_title(name, fontsize=9)
plt.suptitle("Feature Histograms — Raw Data")
plt.tight_layout()
plt.show()

In [ ]:
### Summary statistics
means    = np.mean(raw_data, axis=0)
std_devs = np.std(raw_data, axis=0)
maxes    = np.max(raw_data, axis=0)
mins     = np.min(raw_data, axis=0)
ranges   = np.ptp(raw_data, axis=0)

print(f"{'Column':<12}  {'mean':>10}  {'std':>10}  {'min':>10}  {'max':>10}  {'range':>10}")
print("-" * 62)
for i, name in enumerate(header):
    print(f"{name:<12}  {means[i]:10.3f}  {std_devs[i]:10.3f}  {mins[i]:10.3f}  {maxes[i]:10.3f}  {ranges[i]:10.3f}")

## Step 2 — Choose Preprocessing per Column

Edit the `preproc` list below based on the histograms above:
- `PREP_DROP` — not useful for classification (e.g. timestamp, pressure)
- `PREP_NORM` — non-Gaussian distribution → normalise to `[0, 1]`
- `PREP_STD`  — Gaussian (bell-curve) distribution → standardise to mean=0, σ=1
- `PREP_NONE` — already well-scaled, keep as-is

In [ ]:
### ── Edit this list to match your column order ─────────────────────────────
# Default: timestamp and pressure dropped; everything else normalised
preproc = [
    PREP_DROP,   # timestamp
    PREP_NORM,   # temp
    PREP_NORM,   # humd
    PREP_DROP,   # pres
    PREP_NORM,   # co2
    PREP_NORM,   # voc1
    PREP_NORM,   # voc2
    PREP_NORM,   # no2
    PREP_NORM,   # eth
    PREP_NORM,   # co
]

assert len(preproc) == len(header) == raw_data.shape[1], \
    f"preproc length ({len(preproc)}) must equal number of columns ({len(header)})"
print("Preprocessing plan:")
labels = {PREP_DROP: "DROP", PREP_NONE: "NONE", PREP_STD: "STD", PREP_NORM: "NORM"}
for name, code in zip(header, preproc):
    print(f"  {name:<12} → {labels[code]}")

## Step 3 — Apply Preprocessing

In [ ]:
### Apply preprocessing and collect output statistics

num_cols    = sum(1 for x in preproc if x != PREP_DROP)
prep_data   = np.zeros((raw_data.shape[0], num_cols))
prep_header = []
prep_means  = []
prep_stds   = []
prep_mins   = []
prep_ranges = []

out_col = 0
for raw_col, code in enumerate(preproc):

    if code == PREP_DROP:
        print(f"Dropping '{header[raw_col]}'")
        continue

    elif code == PREP_STD:
        prep_data[:, out_col] = (raw_data[:, raw_col] - means[raw_col]) / std_devs[raw_col]

    elif code == PREP_NORM:
        prep_data[:, out_col] = (raw_data[:, raw_col] - mins[raw_col]) / ranges[raw_col]

    elif code == PREP_NONE:
        prep_data[:, out_col] = raw_data[:, raw_col]

    prep_header.append(header[raw_col])
    prep_means.append(means[raw_col])
    prep_stds.append(std_devs[raw_col])
    prep_mins.append(mins[raw_col])
    prep_ranges.append(ranges[raw_col])
    out_col += 1

print(f"\nPreprocessed data shape: {prep_data.shape}")
print(f"Header : {prep_header}")
print()
print("### Copy these into your inference firmware ###")
print(f"Means  : {[round(x, 4) for x in prep_means]}")
print(f"Std devs: {[round(x, 4) for x in prep_stds]}")
print(f"Mins   : {[round(x, 4) for x in prep_mins]}")
print(f"Ranges : {[round(x, 4) for x in prep_ranges]}")

## Step 4 — Verify Preprocessed Data

In [ ]:
### Scatter matrix of preprocessed data
df_prep = pd.DataFrame(prep_data, columns=prep_header)
_ = pd.plotting.scatter_matrix(df_prep, figsize=(14, 14), alpha=0.3, diagonal="hist")
plt.suptitle("Scatter Matrix — Preprocessed Data", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
### Correlation heatmap of preprocessed data
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.matshow(df_prep.corr(), cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(im)
ax.set_xticks(range(len(prep_header)))
ax.set_xticklabels(prep_header, rotation=45, ha="left")
ax.set_yticks(range(len(prep_header)))
ax.set_yticklabels(prep_header)
ax.set_title("Correlation Matrix — Preprocessed Data", pad=20)
plt.tight_layout()
plt.show()

In [ ]:
### Confirm preprocessed value ranges
print(f"{'Column':<12}  {'min':>8}  {'max':>8}")
print("-" * 32)
for i, name in enumerate(prep_header):
    print(f"{name:<12}  {prep_data[:, i].min():8.4f}  {prep_data[:, i].max():8.4f}")

## Step 5 — Save Preprocessed CSVs and Zip

In [ ]:
### Recreate output directory
if os.path.exists(OUT_PATH):
    shutil.rmtree(OUT_PATH)
os.makedirs(OUT_PATH)
print(f"Output directory: {OUT_PATH}")

In [ ]:
### Write one scaled CSV per original file
row_index = 0
for file_num, filename in enumerate(filenames):
    out_file = os.path.join(OUT_PATH, filename)
    with open(out_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(prep_header)                     # header
        for _ in range(num_lines[file_num]):
            writer.writerow(prep_data[row_index])        # data rows
            row_index += 1
    print(f"  Wrote {num_lines[file_num]:4d} rows → {filename}")

print(f"\nTotal rows written: {row_index}")

In [ ]:
### Zip output directory
%cd {OUT_PATH}
!zip -FS -r -q {OUT_ZIP} *
%cd {HOME_PATH}
print(f"Archive saved: {OUT_ZIP}")
print("Download out.zip and upload the CSVs inside to Edge Impulse.")